# N-gramas y colocaciones

> Cómo encontrar las combinaciones de palabras que funcionan como una unidad (*ice cream*, *customer service*, *Las Vegas*, *highly recommend*): n-gramas y su dispersión, medidas de asociación (frecuencia, PMI, $\chi^2$, razón de verosimilitud), patrones gramaticales y detección automática de términos compuestos con `Phrases` de gensim, sobre reseñas de Yelp. Con los errores típicos: ordenar por PMI sin frecuencia mínima, tomar plantillas con palabras vacías por colocaciones y confundir opiniones con términos.
>
> **Requisitos previos:** [Preprocesamiento de texto](01-preprocesamiento-de-texto.ipynb) · **Relacionado:** [Representación vectorial y TF-IDF](03-representacion-vectorial-tfidf.ipynb) · [Embeddings de palabras](04-embeddings-de-palabras.ipynb)

## 1. Idea clave

Un **n-grama** es una secuencia de $n$ palabras seguidas. Contarlos sirve para dos cosas: estimar qué palabra suele venir después de otra (los **modelos de lenguaje de n-gramas**) y encontrar **colocaciones** (*collocations*): combinaciones que aparecen juntas mucho más de lo que se esperaría por azar y que funcionan como una unidad (*ice cream*, *happy hour*, *go back*).

Detectarlas mejora casi todo lo que viene después: si *ice cream* se convierte en un solo término `ice_cream`, la [bolsa de palabras](03-representacion-vectorial-tfidf.ipynb), los [embeddings](04-embeddings-de-palabras.ipynb) y los [temas](05-deteccion-de-temas.ipynb) dejan de tratarla como «hielo» más «crema». La frecuencia bruta no basta (los pares más frecuentes son *of the*, *it was*), así que se usan **medidas de asociación** que comparan lo observado con lo esperado, combinadas con filtros de palabras vacías, de frecuencia mínima o de categoría gramatical.

## 2. Conceptos importantes

### 2.1. N-gramas y modelos de lenguaje

Con la **hipótesis de Markov** (la probabilidad de una palabra depende solo de las $n-1$ anteriores), la probabilidad de un bigrama se estima por máxima verosimilitud contando:

$$
P(w_i \mid w_{i-1}) = \frac{C(w_{i-1}\,w_i)}{C(w_{i-1})}
$$

donde $C(\cdot)$ es el número de apariciones en el corpus. El problema es la **dispersión** (*sparsity*): por la [ley de Zipf](01-preprocesamiento-de-texto.ipynb), la mayoría de los n-gramas posibles no aparecen nunca o aparecen una vez, y cuanto mayor es $n$, peor. Un n-grama no visto tendría probabilidad 0, por eso los modelos de lenguaje de n-gramas necesitan **suavizado** (*smoothing*: Laplace, Kneser-Ney…), que reserva algo de probabilidad para lo no visto. Hoy los modelos de lenguaje son redes neuronales ([modelos preentrenados](10-modelos-de-lenguaje-preentrenados.ipynb)), pero los recuentos de n-gramas siguen siendo la base para encontrar colocaciones.

### 2.2. Qué es una colocación

Una colocación es una expresión de varias palabras que el uso ha fijado. Suele cumplir alguno de estos criterios:

- **No composicional**: su significado no es la suma de las partes (*hot dog*, *happy hour*).
- **No sustituible**: no se puede cambiar una palabra por un sinónimo (*fast food*, pero no *quick food*).
- **No modificable**: no admite insertar palabras ni cambiar la forma (*ice cream*, no *ice creamy*).

Incluye nombres compuestos (*customer service*, *parking lot*), nombres propios (*Las Vegas*), verbos con partícula (*go back*, *come back*) y expresiones fijas (*bang for your buck*). En cambio, *great food* aparece mucho pero es composicional: *great* se puede cambiar por *good* o *amazing*.

### 2.3. Medidas de asociación

Para un bigrama $x\,y$, las medidas comparan la frecuencia observada con la esperada si $x$ e $y$ fueran **independientes**. Todas parten de la tabla de contingencia de $2\times 2$, donde $O_{11}$ cuenta los bigramas $x\,y$, $O_{12}$ los $x$ seguidos de otra palabra, $O_{21}$ los de otra palabra seguida de $y$ y $O_{22}$ el resto; $N$ es el total de bigramas y $E_{ij}$ el recuento esperado bajo independencia (producto de los totales de fila y columna dividido por $N$).

- **Frecuencia bruta**: $O_{11}$. Premia las combinaciones de palabras frecuentes (*of the*), por eso se usa con filtros de palabras vacías.
- **Información mutua puntual** (PMI, *pointwise mutual information*): cuántas veces más aparece el par de lo esperado, en escala logarítmica:
$$
\operatorname{PMI}(x, y) = \log_2 \frac{P(x\,y)}{P(x)\,P(y)}
$$
  Si $x$ e $y$ solo aparecen una vez y juntas, la PMI es máxima: **favorece los pares raros**, y un solo nombre extranjero la domina. Se usa siempre con una frecuencia mínima.
- **Prueba $t$ de Student**: contrasta si $P(x\,y)$ es mayor que $P(x)P(y)$; en la práctica ordena casi igual que la frecuencia.
- **$\chi^2$ de Pearson**: $\sum_{ij} (O_{ij}-E_{ij})^2 / E_{ij}$. Con recuentos pequeños sobreestima la asociación, como la PMI.
- **Razón de verosimilitud** (*log-likelihood ratio*, Dunning, 1993): compara la verosimilitud de los datos bajo dependencia frente a independencia, $G^2 = 2\sum_{ij} O_{ij}\ln(O_{ij}/E_{ij})$. Es fiable con recuentos pequeños y equilibra frecuencia y asociación; suele ser la mejor opción por defecto.

### 2.4. Otros enfoques

- **Patrones gramaticales**: quedarse solo con secuencias de categorías que forman términos, como adjetivo + nombre (`ADJ NOUN`) o nombre + nombre (`NOUN NOUN`). Necesita un [etiquetador PoS](01-preprocesamiento-de-texto.ipynb) y garantiza que el resultado sea un término nominal, pero no que sea una colocación.
- **`Phrases` de gensim**: une en un solo *token* (`ice_cream`) los bigramas cuya puntuación supera un umbral. Por defecto usa la fórmula de Mikolov et al. (2013), pensada para preparar texto para [Word2Vec](04-embeddings-de-palabras.ipynb):
$$
\text{score}(x, y) = \frac{C(x\,y) - \delta}{C(x)\,C(y)}\cdot \lvert V\rvert
$$
  donde $\delta$ es la frecuencia mínima (`min_count`), que descuenta los pares raros, y $\lvert V\rvert$ el tamaño del vocabulario. Aplicado dos veces, une también expresiones de tres o más palabras (`dulce_de` + `leche`).

## 3. Código

Usamos el mismo corpus que en [preprocesamiento de texto](01-preprocesamiento-de-texto.ipynb): las 50 000 reseñas de prueba de **Yelp Review Full** (Zhang, Zhao y LeCun, 2015), reseñas en inglés de negocios de Yelp (restaurantes, hoteles, talleres…), descargadas de Hugging Face y guardadas en `data/`.

In [1]:
import contextlib
import copy
import html
import io
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import spacy
warnings.filterwarnings("ignore", message="IProgress not found")    # aviso de tqdm al importar datasets
from datasets import load_dataset
from gensim.models.phrases import ENGLISH_CONNECTOR_WORDS, Phrases, npmi_scorer, original_scorer
from huggingface_hub.utils import logging as hf_logging
from nltk.collocations import (BigramAssocMeasures, BigramCollocationFinder,
                               TrigramAssocMeasures, TrigramCollocationFinder)
from nltk.corpus import stopwords

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="NLTK will not authorize")   # aviso de permisos del directorio de NLTK
nltk.download("stopwords", quiet=True);

In [2]:
with contextlib.redirect_stderr(io.StringIO()):          # silencia las barras de descarga
    yelp = load_dataset(
        "Yelp/yelp_review_full",
        data_files={"test": "yelp_review_full/test-00000-of-00001.parquet"},   # solo el conjunto de prueba
        split="test",
        cache_dir="data",
        verification_mode="no_checks",                     # no exige descargar también el de entrenamiento
    )

reviews = yelp.to_pandas()
print(reviews.shape)
reviews["text"].head(3)

(50000, 2)


0    I got 'new' tires from them and within two wee...
1    Don't waste your time.  We had two different p...
2    All I can say is the worst! We were the only 2...
Name: text, dtype: str

### 3.1. Preparación: una lista de *tokens* por oración

Las colocaciones se buscan **dentro de las oraciones**: si se unen todas las reseñas en un solo texto (o se quita la puntuación), aparecen pares artificiales entre el final de una frase y el principio de la siguiente (ver los errores comunes de [preprocesamiento de texto](01-preprocesamiento-de-texto.ipynb)). Limpiamos igual que allí, segmentamos con el `sentencizer` de spaCy (reglas de puntuación, mucho más rápido que el modelo completo) y nos quedamos con los *tokens* alfabéticos en minúsculas.

In [3]:
def clean(text):
    text = text.replace("\\n", " ").replace('\\"', '"')    # secuencias escapadas
    text = html.unescape(text)                              # &amp; -> &
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)      # URL
    return re.sub(r"\s+", " ", text).strip()

reviews["clean"] = reviews["text"].map(clean)

splitter = spacy.blank("en")                               # solo tokenizador ...
splitter.add_pipe("sentencizer")                           # ... y segmentación en oraciones por reglas
sentences = []
for doc in splitter.pipe(reviews["clean"], batch_size=1_000):
    for sent in doc.sents:
        words = [t.lower_ for t in sent if t.is_alpha]
        if len(words) > 1:
            sentences.append(words)

print(f"Oraciones: {len(sentences):,} · tokens: {sum(map(len, sentences)):,}")
print(sentences[0])

Oraciones: 456,322 · tokens: 6,690,544
['i', 'got', 'new', 'tires', 'from', 'them', 'and', 'within', 'two', 'weeks', 'got', 'a', 'flat']


### 3.2. N-gramas: probabilidades y dispersión

Contamos bigramas y trigramas dentro de las oraciones y vemos qué fracción aparece una sola vez. Después separamos el 10 % de las oraciones y medimos cuántos de sus bigramas no aparecen en el 90 % restante: esos son los que un modelo de lenguaje sin suavizado consideraría imposibles.

In [4]:
def ngrams(words, n):
    return zip(*(words[i:] for i in range(n)))

unigram_counts = Counter(w for s in sentences for w in s)
bigram_counts = Counter(b for s in sentences for b in ngrams(s, 2))
trigram_counts = Counter(t for s in sentences for t in ngrams(s, 3))
for name, counts in [("bigramas", bigram_counts), ("trigramas", trigram_counts)]:
    once = sum(c == 1 for c in counts.values())
    print(f"{name}: {len(counts):,} distintos, {100 * once / len(counts):.0f} % aparecen una sola vez")

rng = np.random.default_rng(42)
held_out = rng.random(len(sentences)) < 0.1
train_bigrams = Counter(b for s, h in zip(sentences, held_out) if not h for b in ngrams(s, 2))
test_bigrams = [b for s, h in zip(sentences, held_out) if h for b in ngrams(s, 2)]
unseen = np.mean([b not in train_bigrams for b in test_bigrams])
print(f"Bigramas del 10 % separado que no aparecen en el resto: {100 * unseen:.1f} %")

bigramas: 1,010,886 distintos, 67 % aparecen una sola vez
trigramas: 2,903,273 distintos, 82 % aparecen una sola vez


Bigramas del 10 % separado que no aparecen en el resto: 11.4 %


In [5]:
def next_word_probs(word, k=5):
    # P(w | word) = C(word w) / C(word), estimada con los bigramas dentro de las oraciones
    following = Counter({b: c for (a, b), c in bigram_counts.items() if a == word})
    total = sum(following.values())
    return [f"{w} ({c / total:.2f})" for w, c in following.most_common(k)]

pd.DataFrame({w: next_word_probs(w) for w in ["highly", "customer", "ice", "not"]},
             index=range(1, 6))

,highly,customer,ice,not
1,recommend (0.60),service (0.69),cream (0.60),a (0.06)
2,recommended (0.15),and (0.02),tea (0.05),the (0.04)
3,suggest (0.02),is (0.01),cold (0.04),be (0.03)
4,rated (0.02),for (0.01),and (0.03),sure (0.03)
5,highly (0.02),in (0.01),in (0.02),to (0.03)


- Dos tercios de los bigramas distintos (67 %) y el 82 % de los trigramas aparecen **una sola vez**, y el 11,4 % de los bigramas de oraciones nuevas no se ha visto nunca: sin suavizado, un modelo de bigramas daría probabilidad 0 a más de una de cada diez parejas de palabras de un texto nuevo.
- Las probabilidades condicionadas ya muestran colocaciones: después de *highly* viene *recommend* el 60 % de las veces (y *recommended* el 15 %), después de *customer*, *service* el 69 %, y después de *ice*, *cream* el 60 %. En cambio, después de *not* ninguna palabra pasa del 6 %: no forma parte de una expresión fija.

### 3.3. Colocaciones con medidas de asociación

`BigramCollocationFinder.from_documents` cuenta los bigramas de cada oración por separado (sin crear pares entre oraciones). Quitamos los pares que contienen palabras vacías y comparamos las 10 primeras posiciones según cada medida; la PMI, con y sin frecuencia mínima.

In [6]:
stop_words = set(stopwords.words("english"))
bigram_measures = BigramAssocMeasures()

finder = BigramCollocationFinder.from_documents(sentences)
finder.apply_word_filter(lambda w: w in stop_words)        # descarta pares con alguna palabra vacía
pmi_no_filter = finder.nbest(bigram_measures.pmi, 10)      # antes de exigir frecuencia mínima

finder.apply_freq_filter(10)                               # solo pares que aparecen al menos 10 veces
rankings = {
    "frecuencia": finder.nbest(bigram_measures.raw_freq, 10),
    "PMI (sin mínimo)": pmi_no_filter,
    "PMI (≥ 10)": finder.nbest(bigram_measures.pmi, 10),
    "chi²": finder.nbest(bigram_measures.chi_sq, 10),
    "razón de verosimilitud": finder.nbest(bigram_measures.likelihood_ratio, 10),
}
pd.DataFrame({k: [" ".join(b) for b in v] for k, v in rankings.items()}, index=range(1, 11))

,frecuencia,PMI (sin mínimo),PMI (≥ 10),chi²,razón de verosimilitud
1,go back,aantal sterren,rula bula,rula bula,customer service
2,customer service,abbiamo voluto,dac biet,gon na,las vegas
3,first time,abe froman,knick knacks,pei wei,go back
4,pretty good,accompagnements minimalistes,lomo saltado,wi fi,happy hour
5,come back,accumulent habituellement,puerto rican,dac biet,ice cream
6,las vegas,adeguato porre,pura vida,hong kong,first time
7,next time,adrien brody,celine dion,itty bitty,come back
8,really good,aglia olia,baskin robbins,knick knacks,pretty good
9,happy hour,aglio olio,itty bitty,carne asada,even though
10,even though,aish shul,dom demarco,wal mart,next time


- **Frecuencia** (tras quitar palabras vacías): expresiones habituales del dominio, unas fijas (*customer service*, *Las Vegas*, *ice cream*, *happy hour*) y otras composicionales (*pretty good*, *first time*).
- **PMI sin frecuencia mínima**: todo son pares que aparecen una sola vez, casi todos de reseñas en neerlandés, italiano o francés, o nombres propios raros. Es el comportamiento esperado de la PMI: un par que aparece una vez y cuyas palabras no aparecen en ningún otro sitio tiene la puntuación máxima.
- **PMI y $\chi^2$ con frecuencia mínima 10**: nombres propios, platos y expresiones que casi solo aparecen juntos (*Hong Kong*, *Wal Mart*, *lomo saltado*, *carne asada*, *itty bitty*, *gon na*). Son colocaciones auténticas, pero de baja frecuencia: la lista describe las rarezas del corpus, no sus expresiones típicas.
- **Razón de verosimilitud**: el mejor equilibrio. Las cinco primeras son expresiones frecuentes y fijas (*customer service*, *Las Vegas*, *go back*, *happy hour*, *ice cream*), seguidas de combinaciones composicionales muy frecuentes (*first time*, *pretty good*, *even though*).

Con trigramas pasa lo mismo, con un problema añadido: si solo se filtran las palabras vacías de los extremos, la razón de verosimilitud llena la lista de **plantillas** con una palabra vacía en medio.

In [7]:
trigram_measures = TrigramAssocMeasures()
trigram_finder = TrigramCollocationFinder.from_documents(sentences)
trigram_finder.apply_freq_filter(10)
trigram_finder.apply_ngram_filter(lambda a, b, c: a in stop_words or c in stop_words)   # extremos sin palabras vacías

pd.DataFrame({
    "razón de verosimilitud": [" ".join(t) for t in trigram_finder.nbest(trigram_measures.likelihood_ratio, 8)],
    "PMI (≥ 10)": [" ".join(t) for t in trigram_finder.nbest(trigram_measures.pmi, 8)],
}, index=range(1, 9))

,razón de verosimilitud,PMI (≥ 10)
1,recommend this place,mon ami gabi
2,love this place,pad kee mao
3,give this place,pied de cochon
4,like this place,bun bo hue
5,try this place,nom nom nom
6,found this place,dulce de leche
7,avoid this place,fogo de chao
8,giving this place,chilean sea bass


Las ocho primeras por razón de verosimilitud son variaciones de *___ this place* (*recommend*, *love*, *give*, *avoid*…): una plantilla frecuente, no una unidad léxica. La PMI con frecuencia mínima encuentra nombres de platos y de restaurantes (*pad kee mao*, *dulce de leche*, *Mon Ami Gabi*, *Fogo de Chão*), muchos con una preposición en medio. Para trigramas conviene usar la PMI con un mínimo alto, o el enfoque de `Phrases` (sección 3.5).

### 3.4. Patrones gramaticales

Etiquetamos 5 000 reseñas con spaCy y contamos los pares `NOUN NOUN` y `ADJ NOUN` dentro de la misma reseña (el etiquetador ya trabaja sobre el texto original, con mayúsculas y puntuación).

In [8]:
tagger = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer"])
sample = reviews["clean"].sample(5_000, random_state=42).tolist()

patterns = {"NOUN NOUN": Counter(), "ADJ NOUN": Counter()}
for doc in tagger.pipe(sample, batch_size=256):
    for a, b in zip(doc, doc[1:]):
        key = f"{a.pos_} {b.pos_}"
        if key in patterns and a.is_alpha and b.is_alpha:
            patterns[key][f"{a.lower_} {b.lower_}"] += 1

pd.DataFrame({k: [f"{t} ({c})" for t, c in v.most_common(12)] for k, v in patterns.items()},
             index=range(1, 13))

,NOUN NOUN,ADJ NOUN
1,customer service (232),first time (210)
2,ice cream (99),next time (147)
3,parking lot (53),happy hour (119)
4,dining experience (48),great place (99)
5,wait staff (48),last time (92)
6,credit card (46),only thing (73)
7,crab legs (37),great food (72)
8,beer selection (37),last night (71)
9,bar area (35),good food (68)
10,pork belly (34),mexican food (62)


- `NOUN NOUN` da casi solo **términos compuestos** auténticos: *customer service*, *ice cream*, *parking lot*, *wait staff*, *credit card*. Es el patrón más limpio para extraer aspectos de un producto o servicio.
- `ADJ NOUN` mezcla términos (*happy hour*, *front desk*) con expresiones de tiempo (*first time*, *next time*) y, sobre todo, **opiniones**: *great place*, *great food*, *good food*, *good service*. Son útiles para el [análisis de sentimientos por aspectos](07-analisis-de-sentimientos.ipynb), pero no son colocaciones: el adjetivo es sustituible. Para quedarse con términos hay que combinar el patrón con una medida de asociación.

### 3.5. Detección automática con `Phrases` de gensim

`Phrases` aprende qué bigramas unir y luego transforma cualquier lista de *tokens*. Con `connector_words` puede unir expresiones con palabras gramaticales en medio (*bang_for_your_buck*) sin crear «colocaciones» que empiecen o acaben en ellas. Una segunda pasada sobre el texto ya transformado une trigramas y cuatrigramas.

In [9]:
bigram_model = Phrases(sentences, min_count=10, threshold=10, connector_words=ENGLISH_CONNECTOR_WORDS)
bigram_sentences = list(bigram_model[sentences])
trigram_model = Phrases(bigram_sentences, min_count=10, threshold=10, connector_words=ENGLISH_CONNECTOR_WORDS)

bigram_phrases = bigram_model.export_phrases()             # {frase: puntuación}
all_phrases = trigram_model.export_phrases()
longer = {p: s for p, s in all_phrases.items() if p.count("_") >= 2}
print(f"Frases de 2 palabras: {len(bigram_phrases):,} · frases de 3 o más tras la segunda pasada: {len(longer):,}")

pd.DataFrame({
    "bigramas (mayor puntuación)": sorted(bigram_phrases, key=bigram_phrases.get, reverse=True)[:10],
    "3 o más palabras": sorted(longer, key=longer.get, reverse=True)[:10],
}, index=range(1, 11))

Frases de 2 palabras: 5,085 · frases de 3 o más tras la segunda pasada: 1,154


,bigramas (mayor puntuación),3 o más palabras
1,hong_kong,lotus_of_siam
2,itty_bitty,lo_and_behold
3,lactose_intolerant,chilean_sea_bass
4,panna_cotta,surf_and_turf
5,krispy_kreme,gon_na_lie
6,pei_wei,dulce_de_leche
7,moscow_mule,pico_de_gallo
8,dac_biet,needless_to_say
9,knick_knacks,bait_and_switch
10,wells_fargo,cirque_du_soleil


In [10]:
example = "we went back for happy hour and the customer service was great , they even gave us a free ice cream sandwich".split()
print(trigram_model[bigram_model[example]])

merged = sum(t.count("_") for s in trigram_model[bigram_sentences] for t in s)
print(f"\nPares de palabras unidas en todo el corpus: {merged:,} "
      f"({100 * merged / sum(map(len, sentences)):.1f} % de los tokens)")

['we', 'went', 'back', 'for', 'happy_hour', 'and', 'the', 'customer_service', 'was', 'great', ',', 'they', 'even', 'gave_us', 'a', 'free', 'ice_cream', 'sandwich']



Pares de palabras unidas en todo el corpus: 557,126 (8.3 % de los tokens)


- Los bigramas con mayor puntuación son, como con la PMI, nombres propios y préstamos que casi siempre aparecen juntos (*Hong Kong*, *itty bitty*, *panna cotta*, *Krispy Kreme*). La fórmula descuenta `min_count` del recuento, por eso no aparecen pares de una sola reseña.
- La segunda pasada encuentra expresiones con conectores que ninguna medida de bigramas podía ver: platos (*dulce de leche*, *pico de gallo*, *chilean sea bass*), lugares (*Lotus of Siam*, *Cirque du Soleil*) y frases hechas (*needless to say*, *bait and switch*).
- En la frase de ejemplo une `happy_hour`, `customer_service` e `ice_cream`, que es lo que se quiere antes de entrenar [Word2Vec](04-embeddings-de-palabras.ipynb) o un modelo de [temas](05-deteccion-de-temas.ipynb). Pero también une `gave_us`, que no es ninguna unidad, y deja sueltos *went back* y *ice cream sandwich*: el umbral decide cuánto se une, y no hay un valor que acierte en todo (sección 4).
- El efecto no es pequeño: el 8,3 % de los *tokens* del corpus forma parte de una frase unida.

## 4. Parámetros importantes y errores comunes

| Parámetro | Qué controla | Valores típicos / consejo |
|---|---|---|
| `apply_freq_filter(n)` (NLTK) | Frecuencia mínima del n-grama | Imprescindible con PMI y $\chi^2$: entre 5 y 20 según el tamaño del corpus |
| `apply_word_filter` / `apply_ngram_filter` (NLTK) | Qué candidatos se descartan | Quita los que **empiezan o acaban** en palabra vacía; con trigramas, decide si se admiten en medio (*dulce de leche*) |
| `from_words` frente a `from_documents` (NLTK) | Si los pares cruzan límites de oración o documento | `from_documents` con una lista por oración; `window_size` > 2 cuenta pares no contiguos |
| Medida (`raw_freq`, `pmi`, `chi_sq`, `likelihood_ratio`, `student_t`) | Cómo se ordenan los candidatos | Razón de verosimilitud por defecto; PMI para nombres propios y términos raros, siempre con frecuencia mínima |
| `min_count` (`Phrases`) | Frecuencia mínima y descuento de la puntuación | 5–20; con corpus pequeños, más bajo |
| `threshold` (`Phrases`) | Cuántas frases se unen | Con `scoring="default"` depende del tamaño del corpus (10 por defecto); con `scoring="npmi"` está entre −1 y 1 (0,5 es un buen inicio) y es más fácil de trasladar entre corpus |
| `connector_words` (`Phrases`) | Palabras que pueden ir dentro de una frase pero no en los extremos | `ENGLISH_CONNECTOR_WORDS` para unir *bang_for_your_buck* sin crear *of_the* |

**Errores comunes y lecciones aprendidas**

- **Ordenar por PMI sin frecuencia mínima.** En la sección 3.3 las diez primeras son pares que aparecen una vez. En un análisis propio de reseñas de restaurantes de Yelp pasó lo mismo **incluso después de filtrar por idioma**: la PMI puso arriba *cà phê đá*, *gōngbo jī dīng*, números de calle y cadenas de texto basura, mientras que la razón de verosimilitud daba *ice cream* y *love this place*. La solución es exigir una frecuencia mínima (5–10 apariciones) o usar la razón de verosimilitud.
- **Tomar plantillas por colocaciones.** En ese mismo análisis, las mejores colocaciones de tres palabras por razón de verosimilitud fueron *love this place*, *give this place* y *recommend this place*: plantillas frecuentes con una palabra vacía en medio (sección 3.3). Antes de usarlas como términos, comprueba si se pueden sustituir las palabras.
- **Confundir opiniones con términos.** El patrón `ADJ NOUN` devuelve *great food* o *good service* (sección 3.4), que son valoraciones, no unidades léxicas. Si se quieren aspectos, `NOUN NOUN` o el patrón combinado con una medida de asociación.
- **Elegir el umbral de `Phrases` a ciegas.** Con la puntuación por defecto, el umbral depende del tamaño del corpus y cambia mucho el resultado: con 10 se unen 5 085 frases, entre ellas `gave_us` y `pretty_good`; con 200 se pierden hasta `customer_service` y `happy_hour` (demo abajo). Con `scoring="npmi"` y umbral 0,5 se conservan las expresiones fijas y caen `pretty_good`, `went_back` y `gave_us`; además, su umbral (entre −1 y 1) se traslada mejor de un corpus a otro. En cualquier caso, revisa una muestra de las frases unidas con varios umbrales.
- **Unir todos los documentos en uno.** Crea pares artificiales entre el final de una reseña y el principio de la siguiente; ver la demo en [preprocesamiento de texto](01-preprocesamiento-de-texto.ipynb).

In [11]:
# Cuántas frases se unen y cuáles, según la puntuación y el umbral (se cambian sobre una copia del modelo ya entrenado)
check = ["customer_service", "happy_hour", "ice_cream", "go_back", "first_time", "pretty_good", "went_back", "gave_us"]
model = copy.copy(bigram_model)
rows = {}
for scoring, threshold in [(original_scorer, 5), (original_scorer, 10), (original_scorer, 50), (original_scorer, 200),
                           (npmi_scorer, 0.3), (npmi_scorer, 0.5), (npmi_scorer, 0.7)]:
    model.scoring, model.threshold = scoring, threshold
    phrases = model.export_phrases()
    name = "default" if scoring is original_scorer else "npmi"
    rows[f"{name}, {threshold}"] = {"frases": len(phrases), **{c: "sí" if c in phrases else "" for c in check}}
pd.DataFrame(rows).T

,frases,customer_service,happy_hour,ice_cream,go_back,first_time,pretty_good,went_back,gave_us
"default, 5",7508,sí,sí,sí,sí,sí,sí,sí,sí
"default, 10",5085,sí,sí,sí,sí,sí,sí,,sí
"default, 50",2024,sí,sí,sí,,,,,
"default, 200",944,,,sí,,,,,
"npmi, 0.3",8556,sí,sí,sí,sí,sí,sí,sí,sí
"npmi, 0.5",2202,sí,sí,sí,sí,sí,,,
"npmi, 0.7",526,,sí,sí,,,,,


## 5. Comparación con métodos relacionados

| Método | Qué encuentra | Ventajas | Inconvenientes | Cuándo usarlo |
|---|---|---|---|---|
| Frecuencia + filtro de palabras vacías | Expresiones habituales | Simple, interpretable | Mezcla colocaciones y combinaciones composicionales | Primera exploración del corpus |
| PMI / $\chi^2$ | Pares que casi solo aparecen juntos | Encuentra nombres propios y términos especializados | Dominadas por pares raros: necesitan frecuencia mínima | Terminología y nombres de baja frecuencia |
| Razón de verosimilitud | Pares frecuentes y fuertemente asociados | Fiable con recuentos pequeños; buena opción por defecto | Con trigramas favorece plantillas con palabras vacías | Lista general de colocaciones |
| Patrones PoS (`NOUN NOUN`, `ADJ NOUN`) | Sintagmas nominales | Garantiza términos nominales | Necesita etiquetador; no distingue término de opinión | Extracción de aspectos y terminología |
| `Phrases` de gensim | Frases para unir en un *token* | Transforma el corpus directamente; con conectores, frases largas | El umbral depende del corpus; poco interpretable | Preparar texto para [Word2Vec](04-embeddings-de-palabras.ipynb), [LDA](05-deteccion-de-temas.ipynb) o [TF-IDF](03-representacion-vectorial-tfidf.ipynb) |
| N-gramas como características (`ngram_range`) | Todos los n-gramas, sin seleccionar | Sin decisiones previas | Vocabulario enorme y disperso | Clasificación con [TF-IDF](03-representacion-vectorial-tfidf.ipynb), donde el modelo decide qué n-gramas importan |

## 6. Referencias

### Fuentes

- Zhang, X., Zhao, J. y LeCun, Y. (2015). «Character-level convolutional networks for text classification». *Advances in Neural Information Processing Systems 28 (NIPS 2015)*, 649–657. Origen de Yelp Review Full, descargado de Hugging Face ([`Yelp/yelp_review_full`](https://huggingface.co/datasets/Yelp/yelp_review_full)) con [`datasets.load_dataset`](https://huggingface.co/docs/datasets/loading).
- NLTK: [Collocations HOWTO](https://www.nltk.org/howto/collocations.html), [`nltk.collocations`](https://www.nltk.org/api/nltk.collocations.html) y [`nltk.metrics.association`](https://www.nltk.org/api/nltk.metrics.association.html) (medidas de asociación).
- gensim: [`models.phrases`](https://radimrehurek.com/gensim/models/phrases.html) (`Phrases`, fórmulas de puntuación y `ENGLISH_CONNECTOR_WORDS`).
- spaCy: [`Sentencizer`](https://spacy.io/api/sentencizer) y [Linguistic Features](https://spacy.io/usage/linguistic-features) (etiquetado PoS).

### Para saber más

- Manning, C. D. y Schütze, H. (1999). [*Foundations of Statistical Natural Language Processing*](https://nlp.stanford.edu/fsnlp/). MIT Press. El capítulo 5 (*Collocations*) es la referencia clásica del tema: criterios lingüísticos, frecuencia con patrones PoS, prueba $t$, $\chi^2$, razón de verosimilitud y PMI con ejemplos; el 6, los modelos de n-gramas y el suavizado.
- Jurafsky, D. y Martin, J. H. (2026). [*Speech and Language Processing*](https://web.stanford.edu/~jurafsky/slp3/) (3.ª ed., borrador en línea). El capítulo 3 (*N-gram Language Models*) explica los modelos de n-gramas, su evaluación con la perplejidad y el suavizado.